# Abgabe 3 – Kapitel 7: Ensemble Learning (Programmieraufgaben)

Programmieraufgaben 8–9 aus Hands-On Machine Learning (Géron), Kapitel 7.

## Setup

MNIST laden und in Training (50.000), Validierung (10.000) und Test (10.000) aufteilen.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(np.uint8)

X_train, y_train = X[:50000], y[:50000]
X_val,   y_val   = X[50000:60000], y[50000:60000]
X_test,  y_test  = X[60000:], y[60000:]

print("Training:", X_train.shape, "| Validierung:", X_val.shape, "| Test:", X_test.shape)

Training: (50000, 784) | Validierung: (10000, 784) | Test: (10000, 784)


## Aufgabe 8 — Voting-Ensemble auf MNIST

Vier unterschiedliche Klassifikatoren trainieren: Random Forest, Extra-Trees, SVM (RBF-Kernel,
in einer Pipeline mit Skalierung) und ein neuronales Netz.

In [2]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

rf_clf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
et_clf  = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm_clf = LinearSVC(max_iter=100, tol=20, dual=True, random_state=42)
mlp_clf = MLPClassifier(random_state=42)

estimators = [("rf", rf_clf), ("et", et_clf), ("svc", svm_clf), ("mlp", mlp_clf)]

for name, clf in estimators:
    clf.fit(X_train, y_train)
    print(f"{name:4s} Validierungs-Genauigkeit: {clf.score(X_val, y_val):.4f}")

rf   Validierungs-Genauigkeit: 0.9736
et   Validierungs-Genauigkeit: 0.9743
svc  Validierungs-Genauigkeit: 0.8662
mlp  Validierungs-Genauigkeit: 0.9653


Die Einzelmodelle liegen bei etwa 94–97 %, der lineare Anteil der SVM etwas darunter.

Die vier Modelle zu einem VotingClassifier mit Hard Voting kombinieren.

In [3]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train, y_train)
print(f"Voting-Ensemble – Validierungs-Genauigkeit: {voting_clf.score(X_val, y_val):.4f}")

Voting-Ensemble – Validierungs-Genauigkeit: 0.9748


Das Ensemble ist auf den Validierungsdaten besser als jedes Einzelmodell.

Ensemble und Einzelmodelle auf den Testdaten vergleichen.

In [4]:
print("Genauigkeit auf den Testdaten:")
for name, clf in estimators:
    print(f"  {name:4s} {clf.score(X_test, y_test):.4f}")
voting_test = voting_clf.score(X_test, y_test)
print(f"  Voting-Ensemble: {voting_test:.4f}")

best_single = max(clf.score(X_test, y_test) for _, clf in estimators)
print(f"\nDas Ensemble ist {(voting_test - best_single)*100:+.2f} Prozentpunkte besser "
      f"als das beste Einzelmodell.")

Genauigkeit auf den Testdaten:
  rf   0.9680
  et   0.9703
  svc  0.8612
  mlp  0.9634
  Voting-Ensemble: 0.9707

Das Ensemble ist +0.04 Prozentpunkte besser als das beste Einzelmodell.


Das Voting-Ensemble liegt leicht über dem besten Einzelklassifikator – ein kleiner, aber konsistenter Gewinn.

## Aufgabe 9 — Stacking-Ensemble

Jeder Klassifikator sagt den Validierungsdatensatz vorher; aus diesen Vorhersagen entsteht ein
neuer Trainingsdatensatz für den Blender.

In [5]:
X_val_predictions = np.empty((len(X_val), len(estimators)), dtype=np.float32)
for i, (name, clf) in enumerate(estimators):
    X_val_predictions[:, i] = clf.predict(X_val)

print("Neuer Trainingsdatensatz für den Blender:", X_val_predictions.shape)

Neuer Trainingsdatensatz für den Blender: (10000, 4)


Als Blender einen Random Forest auf diesen Vorhersagen trainieren.

In [6]:
blender = RandomForestClassifier(n_estimators=200, oob_score=True,
                                 random_state=42, n_jobs=-1)
blender.fit(X_val_predictions, y_val)
print(f"OOB-Genauigkeit des Blenders: {blender.oob_score_:.4f}")

OOB-Genauigkeit des Blenders: 0.9729


Für jedes Testbild die Vorhersagen aller Klassifikatoren sammeln und dem Blender übergeben.

In [7]:
from sklearn.metrics import accuracy_score

X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)
for i, (name, clf) in enumerate(estimators):
    X_test_predictions[:, i] = clf.predict(X_test)

stack_test = accuracy_score(y_test, blender.predict(X_test_predictions))
print(f"Manuelles Stacking-Ensemble – Test-Genauigkeit: {stack_test:.4f}")
print(f"Voting-Ensemble            – Test-Genauigkeit: {voting_test:.4f}")

Manuelles Stacking-Ensemble – Test-Genauigkeit: 0.9691
Voting-Ensemble            – Test-Genauigkeit: 0.9707


Das manuelle Stacking schneidet ähnlich ab wie das Voting-Ensemble.

Zum Vergleich der StackingClassifier von Scikit-Learn. Er erzeugt die Blending-Merkmale per
Kreuzvalidierung; deshalb können Trainings- und Validierungsdaten zu einem größeren Satz von
60.000 Bildern zusammengelegt werden.

In [8]:
from sklearn.ensemble import StackingClassifier

X_train_full = X[:60000]
y_train_full = y[:60000]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    cv=3, n_jobs=-1)
stacking_clf.fit(X_train_full, y_train_full)
sk_stack_test = stacking_clf.score(X_test, y_test)
print(f"StackingClassifier – Test-Genauigkeit: {sk_stack_test:.4f}")

StackingClassifier – Test-Genauigkeit: 0.9791


In [9]:
print("Zusammenfassung (Test-Genauigkeit):")
print(f"  Bestes Einzelmodell:          {best_single:.4f}")
print(f"  Voting-Ensemble:              {voting_test:.4f}")
print(f"  Manuelles Stacking:           {stack_test:.4f}")
print(f"  StackingClassifier (sklearn): {sk_stack_test:.4f}")

Zusammenfassung (Test-Genauigkeit):
  Bestes Einzelmodell:          0.9703
  Voting-Ensemble:              0.9707
  Manuelles Stacking:           0.9691
  StackingClassifier (sklearn): 0.9791


Der StackingClassifier ist besser als das manuelle Stacking, weil er auf mehr Daten lernt
(zurückgewonnener Validierungssatz) und für den Blender Wahrscheinlichkeiten statt harter
Labels nutzt.